# Question Answering with BERT on SQuAD

This notebook walks through the first stage of building a **Question Answering (QA)** system using the **SQuAD** dataset and a **BERT** tokenizer.

**What this notebook covers:**
1. Loading and exploring the SQuAD dataset
2. Inspecting the structure of a single QA example
3. Converting the dataset to a pandas DataFrame for exploratory analysis
4. Tokenizing question/context pairs with `BertTokenizerFast`
5. Understanding `input_ids`, `attention_mask`, and `token_type_ids`
6. Applying padding and truncation for fixed-length model inputs

---


## 1. Imports

Core libraries for data handling, numerical operations, visualization, and dataset loading.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset

## 2. Load the SQuAD Dataset

We use the Hugging Face `datasets` library to load the **SQuAD** (Stanford Question Answering Dataset) benchmark, which contains context passages paired with questions and their answers.

In [2]:
dataset = load_dataset("squad")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Let's inspect the overall structure of the dataset (splits and features available).

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

## 3. Explore a Single Example

Each example in SQuAD consists of an `id`, `title`, `context` paragraph, a `question`, and the `answers` (text + starting character index).

In [4]:
dataset["train"][0]

{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

### 3.1 Inspect Individual Fields

In [5]:
print(dataset["train"][0]["context"])

Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.


In [6]:
print(dataset["train"][0]["question"])

To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?


In [7]:
print(dataset["train"][0]["answers"])

{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


### 3.2 Extract and Display Question / Answer / Answer Start

We pull out the individual fields of a sample for clarity.

In [8]:
sample = dataset["train"][0]

context = sample["context"]
question = sample["question"]
answer = sample["answers"]["text"][0]
start = sample["answers"]["answer_start"][0]

print("Question : ", question)
print()
print("Answer   : ", answer)
print()
print("Start    : ", start)

Question :  To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

Answer   :  Saint Bernadette Soubirous

Start    :  515


### 3.3 Verify the Answer Span

Using the `answer_start` index, we slice the context to confirm it matches the reported answer text.

In [9]:
end = start + len(answer)

print(context[start:end])

Saint Bernadette Soubirous


We can also visualize the answer *in context* by splitting the passage into the text before, during, and after the answer span.

In [10]:
print(context[:start])
print(context[start:end])
print(context[end:])

Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to 
Saint Bernadette Soubirous
 in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.


## 4. Dataset Size

Check the number of examples available in each split.

In [11]:
len(dataset["train"])

87599

In [12]:
len(dataset["validation"])

10570

## 5. Convert to a pandas DataFrame

Converting the training split to a DataFrame makes it easier to explore the data using familiar pandas tooling.

In [13]:
train_df = pd.DataFrame(dataset["train"])

train_df.head()

,id,title,context,question,answers
0,5733be284776f41900661182,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ..."
1,5733be284776f4190066117f,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,"{'text': ['a copper statue of Christ'], 'answe..."
2,5733be284776f41900661180,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,"{'text': ['the Main Building'], 'answer_start'..."
3,5733be284776f41900661181,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,{'text': ['a Marian place of prayer and reflec...
4,5733be284776f4190066117e,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,{'text': ['a golden statue of the Virgin Mary'...


### 5.1 Data Types and Non-Null Counts

In [14]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87599 entries, 0 to 87598
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        87599 non-null  object
 1   title     87599 non-null  object
 2   context   87599 non-null  object
 3   question  87599 non-null  object
 4   answers   87599 non-null  object
dtypes: object(5)
memory usage: 3.3+ MB


### 5.2 Check for Missing Values

In [15]:
train_df.isnull().sum()

id          0
title       0
context     0
question    0
answers     0
dtype: int64

## 6. Tokenization with BERT

To feed text into a BERT-based QA model, we need to tokenize the question/context pairs using the model's matching tokenizer.

In [16]:
from transformers import BertTokenizerFast

Load the pretrained `bert-base-uncased` tokenizer.

In [17]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

### 6.1 Prepare a Sample Question/Context Pair

In [18]:
sample = dataset["train"][0]

question = sample["question"]
context = sample["context"]

### 6.2 Tokenize the Pair

BERT expects the question and context to be passed together, separated internally by special tokens (`[CLS]`, `[SEP]`).

In [19]:
encoding = tokenizer(question, context)

Inspect the keys returned by the tokenizer.

In [20]:
encoding.keys()

KeysView({'input_ids': [101, 2000, 3183, 2106, 1996, 6261, 2984, 9382, 3711, 1999, 8517, 1999, 10223, 26371, 2605, 1029, 102, 6549, 2135, 1010, 1996, 2082, 2038, 1037, 3234, 2839, 1012, 10234, 1996, 2364, 2311, 1005, 1055, 2751, 8514, 2003, 1037, 3585, 6231, 1997, 1996, 6261, 2984, 1012, 3202, 1999, 2392, 1997, 1996, 2364, 2311, 1998, 5307, 2009, 1010, 2003, 1037, 6967, 6231, 1997, 4828, 2007, 2608, 2039, 14995, 6924, 2007, 1996, 5722, 1000, 2310, 3490, 2618, 4748, 2033, 18168, 5267, 1000, 1012, 2279, 2000, 1996, 2364, 2311, 2003, 1996, 13546, 1997, 1996, 6730, 2540, 1012, 3202, 2369, 1996, 13546, 2003, 1996, 24665, 23052, 1010, 1037, 14042, 2173, 1997, 7083, 1998, 9185, 1012, 2009, 2003, 1037, 15059, 1997, 1996, 24665, 23052, 2012, 10223, 26371, 1010, 2605, 2073, 1996, 6261, 2984, 22353, 2135, 2596, 2000, 3002, 16595, 9648, 4674, 2061, 12083, 9711, 2271, 1999, 8517, 1012, 2012, 1996, 2203, 1997, 1996, 2364, 3298, 1006, 1998, 1999, 1037, 3622, 2240, 2008, 8539, 2083, 1017, 11342, 1998,

### 6.3 Convert Token IDs Back to Tokens

This helps visualize how BERT's WordPiece tokenizer splits the text.

In [21]:
tokens = tokenizer.convert_ids_to_tokens(
    encoding["input_ids"]
)

print(tokens)

['[CLS]', 'to', 'whom', 'did', 'the', 'virgin', 'mary', 'allegedly', 'appear', 'in', '1858', 'in', 'lou', '##rdes', 'france', '?', '[SEP]', 'architectural', '##ly', ',', 'the', 'school', 'has', 'a', 'catholic', 'character', '.', 'atop', 'the', 'main', 'building', "'", 's', 'gold', 'dome', 'is', 'a', 'golden', 'statue', 'of', 'the', 'virgin', 'mary', '.', 'immediately', 'in', 'front', 'of', 'the', 'main', 'building', 'and', 'facing', 'it', ',', 'is', 'a', 'copper', 'statue', 'of', 'christ', 'with', 'arms', 'up', '##rai', '##sed', 'with', 'the', 'legend', '"', 've', '##ni', '##te', 'ad', 'me', 'om', '##nes', '"', '.', 'next', 'to', 'the', 'main', 'building', 'is', 'the', 'basilica', 'of', 'the', 'sacred', 'heart', '.', 'immediately', 'behind', 'the', 'basilica', 'is', 'the', 'gr', '##otto', ',', 'a', 'marian', 'place', 'of', 'prayer', 'and', 'reflection', '.', 'it', 'is', 'a', 'replica', 'of', 'the', 'gr', '##otto', 'at', 'lou', '##rdes', ',', 'france', 'where', 'the', 'virgin', 'mary', 

### 6.4 Map Each Token to Its ID

In [22]:
for token, id in zip(tokens, encoding["input_ids"]):
    print(f"{token:15} ---> {id}")

[CLS]           ---> 101
to              ---> 2000
whom            ---> 3183
did             ---> 2106
the             ---> 1996
virgin          ---> 6261
mary            ---> 2984
allegedly       ---> 9382
appear          ---> 3711
in              ---> 1999
1858            ---> 8517
in              ---> 1999
lou             ---> 10223
##rdes          ---> 26371
france          ---> 2605
?               ---> 1029
[SEP]           ---> 102
architectural   ---> 6549
##ly            ---> 2135
,               ---> 1010
the             ---> 1996
school          ---> 2082
has             ---> 2038
a               ---> 1037
catholic        ---> 3234
character       ---> 2839
.               ---> 1012
atop            ---> 10234
the             ---> 1996
main            ---> 2364
building        ---> 2311
'               ---> 1005
s               ---> 1055
gold            ---> 2751
dome            ---> 8514
is              ---> 2003
a               ---> 1037
golden          ---> 3585
statue     

### 6.5 Token Type IDs

`token_type_ids` distinguish the question (`0`) from the context (`1`) within the combined sequence.

In [23]:
encoding["token_type_ids"]

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

### 6.6 Attention Mask

The `attention_mask` marks which tokens are real input (`1`) versus padding (`0`). Since no padding was applied yet, all values are `1`.

In [24]:
encoding["attention_mask"]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

## 7. Padding and Truncation

Models require fixed-length inputs within a batch. We re-tokenize the pair with `padding="max_length"` and `truncation=True` to enforce a fixed sequence length of 50 tokens.

In [25]:
encoding = tokenizer(
    question,
    context,
    padding="max_length",
    max_length=50,
    truncation=True
)

Now the `attention_mask` reflects the padded tokens (`0`) added at the end of the sequence.

In [26]:
encoding["attention_mask"]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

### 7.1 Full Encoding Summary

Print `input_ids`, `attention_mask`, and `token_type_ids` together for a complete view of the padded/truncated encoding.

In [27]:
print("Input IDs")
print(encoding["input_ids"])
print()

print("Attention Mask")
print(encoding["attention_mask"])
print()

print("Token Type IDs")
print(encoding["token_type_ids"])

Input IDs
[101, 2000, 3183, 2106, 1996, 6261, 2984, 9382, 3711, 1999, 8517, 1999, 10223, 26371, 2605, 1029, 102, 6549, 2135, 1010, 1996, 2082, 2038, 1037, 3234, 2839, 1012, 10234, 1996, 2364, 2311, 1005, 1055, 2751, 8514, 2003, 1037, 3585, 6231, 1997, 1996, 6261, 2984, 1012, 3202, 1999, 2392, 1997, 1996, 102]

Attention Mask
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Token Type IDs
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 8. Next Steps

With tokenization and encoding understood, the following steps would typically follow:

- Map answer character spans to **token start/end positions** for training a QA head
- Batch-tokenize the full dataset with consistent padding/truncation
- Fine-tune a `BertForQuestionAnswering` model on the encoded dataset
- Evaluate using SQuAD's Exact Match (EM) and F1 metrics


## 9. Preprocessing the Dataset for Training

To train a QA model, each example needs to be converted into:
- Tokenized `input_ids` / `attention_mask` / `token_type_ids`
- A **start position** and **end position** (token indices) marking where the answer lies inside the tokenized sequence

We use `return_offsets_mapping=True` to map character-level answer spans to token-level positions, and a `stride` with `return_overflowing_tokens=True` to correctly handle contexts longer than `max_length`.

In [28]:
MAX_LENGTH = 384
STRIDE = 128

def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]

    encodings = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = encodings.pop("overflow_to_sample_mapping")
    offset_mapping = encodings.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = examples["answers"][sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = encodings.sequence_ids(i)

        # Find the start and end of the context within the tokenized sequence
        context_start = sequence_ids.index(1)
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1)

        # If the answer is not fully inside the context window, label it (0, 0)
        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    encodings["start_positions"] = start_positions
    encodings["end_positions"] = end_positions

    return encodings

Apply the preprocessing function across the dataset in batches, removing the original text columns since the model only needs the encoded tensors.

In [29]:
tokenized_train = dataset["train"].map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

tokenized_validation = dataset["validation"].map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["validation"].column_names,
)

tokenized_train, tokenized_validation

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

(Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
     num_rows: 88524
 }),
 Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
     num_rows: 10784
 }))

## 10. Build the QA Model

We use `BertForQuestionAnswering`, which adds a span-prediction head (two linear layers predicting the start and end token of the answer) on top of the base BERT encoder.

In [30]:
from transformers import BertForQuestionAnswering

model = BertForQuestionAnswering.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

## 11. Fine-Tune the Model

We use the Hugging Face `Trainer` API to handle the training loop, evaluation, and checkpointing.

> **Note:** Full training on all ~88K SQuAD examples is compute-intensive and best run on a GPU. For demonstration purposes, this notebook trains on a smaller subset. Remove the `.select(...)` calls below (and increase `num_train_epochs`) to train on the full dataset with a GPU-enabled environment.

In [31]:
from transformers import TrainingArguments, Trainer, DefaultDataCollator

# Use a smaller subset for a quick demonstration run.
# Remove .select(...) to train on the full dataset.
train_subset = tokenized_train.select(range(2000))
eval_subset = tokenized_validation.select(range(500))

data_collator = DefaultDataCollator()

training_args = TrainingArguments(
    output_dir="./qa-model-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
)

In [35]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
    data_collator=data_collator,
)

In [36]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,2.279046,2.161752


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=3.1979518737792967, metrics={'train_runtime': 5181.5387, 'train_samples_per_second': 0.386, 'train_steps_per_second': 0.048, 'total_flos': 391945135104000.0, 'train_loss': 3.1979518737792967, 'epoch': 1.0})

### 11.1 Save the Fine-Tuned Model

In [37]:
trainer.save_model("./qa-model")
tokenizer.save_pretrained("./qa-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./qa-model/tokenizer_config.json', './qa-model/tokenizer.json')

## 12. Prediction Test

Now we test the fine-tuned model on a real question/context pair to see the predicted answer.

### 12.1 Manual Prediction (Raw Logits)

We tokenize a question/context pair, run it through the model, and decode the tokens between the predicted start and end positions.

In [38]:
import torch

test_sample = dataset["validation"][0]
test_question = test_sample["question"]
test_context = test_sample["context"]

inputs = tokenizer(
    test_question,
    test_context,
    return_tensors="pt",
    truncation="only_second",
    max_length=MAX_LENGTH,
)

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits)

predicted_tokens = inputs["input_ids"][0][start_idx : end_idx + 1]
predicted_answer = tokenizer.decode(predicted_tokens, skip_special_tokens=True)

print("Question         :", test_question)
print("Context           :", test_context)
print()
print("Predicted Answer  :", predicted_answer)
print("Ground Truth      :", test_sample["answers"]["text"][0])

Question         : Which NFL team represented the AFC at Super Bowl 50?
Context           : Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.

Predicted Answer  : 
Ground Truth      : Denver Broncos


### 12.2 Prediction Using the `pipeline` API

For convenience, Hugging Face's `pipeline` wraps tokenization, inference, and decoding into a single call — this is the recommended way to run predictions in production.

In [39]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
)

result = qa_pipeline(question=test_question, context=test_context)
result

{'score': 0.051536522805690765,
 'start': 177,
 'end': 191,
 'answer': 'Denver Broncos'}

### 12.3 Try a Custom Example

In [40]:
custom_context = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars "
    "in Paris, France. It was designed by the engineer Gustave Eiffel and "
    "completed in 1889 as the entrance arch to the World's Fair."
)
custom_question = "Who designed the Eiffel Tower?"

qa_pipeline(question=custom_question, context=custom_context)

{'score': 0.23982292413711548,
 'start': 120,
 'end': 134,
 'answer': 'Gustave Eiffel'}